# Forelesning 8: Differensiallikninger og Eulers metode

I dag skal vi

- se hvordan en fartslov er en differensiallikning
- utlede og programmere **Eulers metode**
- kontrollere simuleringen mot den analytiske løsningen
- undersøke hva tidssteget betyr for feil og stabilitet

In [2]:
import numpy as np
import matplotlib.pyplot as plt

## Del 1: Hva er en differensiallikning?

For en førsteordens reaksjon A → produkter er fartsloven

$$\frac{d[\mathrm{A}]}{dt} = -k[\mathrm{A}]$$

Fartsloven sier hvor raskt konsentrasjonen endrer seg **akkurat nå**. Vi ønsker å finne hele utviklingen $[\mathrm{A}](t)$.

<div class="viktig">

En **differensiallikning** inneholder en ukjent funksjon og den deriverte til funksjonen, for eksempel

$$\frac{dy}{dt} = f(t, y)$$

Venstresiden er **endringen**, og høyresiden forteller hva endringen avhenger av. Løsningen er en **funksjon**, ikke et tall.

</div>

<div class="diskuter">

**Diskuter:** Differensiallikningen $y' = 1$ har løsningen $y = t + C$. Hvor mange løsninger finnes det? Hva trenger vi mer for å få én bestemt løsning, og hva tilsvarer det i et kjemisk forsøk?

</div>

<div class="viktig">

Vi trenger en **initialbetingelse**, for eksempel $y(0) = 2$. I kinetikk er det startkonsentrasjonen $[\mathrm{A}]_0$.

</div>

## Del 2: Eulers metode

<div class="bilde-hoyre" style="width:42%">

![Forward Euler og Backward Euler](bilder/forward_backward_euler.png)
<p class="kreditering">Forward Euler og Backward Euler</p>

</div>

**Idé:** Vi kjenner startverdien og stigningen. Vi går et lite tidssteg $\Delta t$ i retning av stigningen, beregner den nye stigningen og gjentar.

Vi utleder metoden på tavla fra framoverdifferansen i numerisk derivasjon.

<div class="viktig">

**Eulers metode** (Forward Euler)

$$y_{n+1} = y_n + f(t_n, y_n)\cdot\Delta t$$

Ny verdi = gammel verdi + stigning · tidssteg

</div>

### Eulers metode grafisk

<div class="bilde-midt" style="width:70%">

![Euler-steg sammenliknet med eksakt løsning](bilder/euler_steg.png)

</div>

<div class="diskuter">

**Diskuter:** Hvorfor havner Euler-løsningen under den eksakte løsningen her? Hva skjer hvis vi gjør tidssteget mindre?

</div>

## Del 3: Førsteordens reaksjon i Python

$k = 0{,}030\ \mathrm{s^{-1}}$ og $[\mathrm{A}]_0 = 1{,}00$ mol/L.

<div class="diskuter">

**Diskuter:** Hvilken linje i programmet *er* Eulers metode? Hva gjør resten av linjene?

</div>

<div class="oppgave">

**Prøv selv:** Den analytiske løsningen er $[\mathrm{A}](t) = [\mathrm{A}]_0 e^{-kt}$. Plott den i samme figur som Euler-løsningen, og beregn den største absolutte feilen.

</div>

In [ ]:
times = np.array(times)
conc = np.array(conc)
conc_exact = ??

plt.plot(times, conc, label="Euler")
plt.plot(times, conc_exact, "--", label="Exact")
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.legend()
plt.show()

print("Largest error:", ??)

## Del 4: Eulers metode som funksjon

Vi skiller **metoden** fra **den kjemiske modellen**. Fartsloven skrives som en funksjon `f(t, y)`, slik at den samme Euler-funksjonen kan brukes på alle fartslover.

<div class="oppgave">

**Prøv selv:** Fyll inn linja som utfører Euler-steget. Test funksjonen på førsteordensreaksjonen.

</div>

In [ ]:
def euler(f, y0, t0, t_end, dt):
    t = t0
    y = y0
    times = [t]
    values = [y]
    while t < t_end:
        y = ??
        t = t + dt
        times.append(t)
        values.append(y)
    return np.array(times), np.array(values)

def first_order(t, A):
    return -k * A

times, conc = euler(first_order, 1.00, 0, 150, 1.0)
print(conc[-1], np.exp(-k * times[-1]))

## Del 5: Hvor stort tidssteg?

Halver tidssteget flere ganger, og **fyll inn** feilen ved $t = 150$ s. Hva skjer med feilen når $\Delta t$ halveres?

In [ ]:
for dt in [10, 5, 2.5, 1.25]:
    times, conc = euler(first_order, 1.00, 0, 150, dt)
    error = ??
    print(f"dt = {dt:5.2f} s   error = {error:.2e}")

<div class="viktig">

Euler er en **førsteordens metode**: Halverer vi $\Delta t$, blir feilen omtrent halvert. Kontroller alltid en simulering ved å redusere tidssteget og se om løsningen endrer seg.

</div>

### For store tidssteg

<div class="diskuter">

**Tipp først:** Hva skjer hvis tidssteget blir **stort**, for eksempel 40 s eller 70 s, når $k = 0{,}030\ \mathrm{s^{-1}}$? Kjør koden.

</div>

In [ ]:
for dt in [10, 40, 70]:
    times, conc = euler(first_order, 1.00, 0, 150, dt)
    plt.plot(times, conc, "o-", label=f"dt = {dt} s")

plt.axhline(0, color="gray")
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.legend()
plt.show()

<div class="viktig">

Med for stort tidssteg tar Euler et for langt steg langs stigningen og skyter over null. Da får vi **negative konsentrasjoner** og svingninger. Løsningen er **ustabil** og kjemisk meningsløs. For denne fartsloven skjer det når $k\Delta t > 2$.

</div>

<div class="diskuter">

**Diskuter:** Hvorfor er det ikke nok at programmet kjører uten feilmelding? Hvilke kjemiske kontroller kan du gjøre på en simulering?

</div>

## Del 6: Andreordens reaksjon

For 2A → produkter er fartsloven $\dfrac{d[\mathrm{A}]}{dt} = -k[\mathrm{A}]^2$, med analytisk løsning $\dfrac{1}{[\mathrm{A}]} = \dfrac{1}{[\mathrm{A}]_0} + kt$.

<div class="oppgave">

**Prøv selv:** Skriv fartsloven som en funksjon, og bruk `euler` med $k = 0{,}50$ L/(mol·s), $[\mathrm{A}]_0 = 1{,}00$ mol/L og $\Delta t = 0{,}5$ s fram til $t = 20$ s. Sammenlikn med den analytiske løsningen i samme figur.

</div>

In [ ]:
def second_order(t, A):
    return ??

times, conc = euler(??)
conc_exact = ??

plt.plot(times, conc, label="Euler")
plt.plot(times, conc_exact, "--", label="Exact")
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.legend()
plt.show()

<div class="diskuter">

**Diskuter:** Sammenlikn med førsteordensreaksjonen. Hvorfor avtar konsentrasjonen langsommere mot slutten for en andreordens reaksjon?

</div>

__Oppgaver__:
1. Løs $d[\mathrm{A}]/dt = -0{,}20[\mathrm{A}]$ med $[\mathrm{A}]_0 = 1{,}50$ mol/L med Euler. Sammenlikn med den analytiske løsningen etter 5, 10 og 20 s.
2. Gjenta oppgave 1 med $\Delta t$ = 2,0, 1,0, 0,5, 0,1 og 0,01 s. Plott feilen ved $t = 20$ s mot $\Delta t$ med logaritmiske akser. Hvilken stigning har linja, og hva betyr det?
3. Halveringstiden for en førsteordens reaksjon er $t_{1/2} = \ln 2 / k$. Finn halveringstiden fra Euler-simuleringen, for eksempel med `np.argmax(conc <= 0.5)`, og sammenlikn med formelen.
4. Hva er det største tidssteget som gir en stabil løsning for $k = 0{,}20\ \mathrm{s^{-1}}$? Test numerisk og forklar.
5. Newtons avkjølingslov sier at $dT/dt = -k(T - T_{\mathrm{omg}})$. Simuler avkjølingen av en kopp kaffe fra 85 °C i et rom på 21 °C med $k = 0{,}05\ \mathrm{min^{-1}}$. Når er kaffen under 50 °C?